# Dynamic Shape CUDA Graphs with Triton

**Background:** this is a short example of how dynamic-shape CUDA graphs can trace through Triton, detect guards, and replay at varying sizes and memory addresses.

We start from a standard user Triton kernel. From there, we need three things:

- A tracer that runs user code with symbolic shapes and memory addresses, and records guards.
- Introspection into Triton, to determine which guards the compiled kernel relies on and how our symbolic arguments map to kernel parameters in the CUDA graph.
- A runtime that checks guards, does memory allocations, computes the size arithmetic, and then updates the CUDA graph's parameters.

We intentionally don't handle Triton's full feature set, to keep this concise. In the future we'll work with the Triton team on the best, least invasive APIs Triton could expose for the same functionality.

While we only handle Triton here, the same approach applies to C++ CUDA launches and CuTe DSL kernels: run the host code with symbolic sizes and addresses, guard on what the kernel compiles in, and map the arguments to kernel parameters.

The first call at new inputs traces the user code: it runs it once as a warm-up (which compiles what these inputs need and gives the call's result), then runs it again under CUDA graph capture, with the same inputs mapped to symbolic sizes. Later calls whose guards hold replay that graph.

Each guard has an owner, which decides what happens when it fails:

- **The graph** owns a guard the user code's Python records, such as a branch on a size. If it fails, the user code would run differently, so the call is traced again.
- **A launch** owns a guard that Triton's dispatch of that launch records: a constexpr computed from a size, and each check Triton's binder makes on an argument (divisible by 16, equal to 1, in i32 range). If one fails, only that launch's dispatch runs again, at the call's values, and the compilation it picks is captured as another entry of the same graph. No user code runs, and nothing else is traced.

Four files, each written by a cell:

1. `user_code.py`: the user code, untouched
2. `triton_hook.py`: the only code that touches Triton
3. `tracing.py`: the tracer
4. `replay.py`: the runtime; `DynamicCudaGraph` puts trace and replay together

Section 6 runs the same calls through our full implementation, for comparison.

Needs a GPU, Triton (3.8 here) and `cuda.bindings`. Section 6 also needs our host-tracing build. The outputs are from a GB300, on our build of 2026-10-08 (internal snapshot "candidate 3").

## 1. User kernel

A plain Triton add. The host code picks the block size from `n` with a Python branch, and passes `INT64=n >= 2**31` as a constexpr, so which kernel eager compiles depends on the size.

In [1]:
%%writefile user_code.py
import torch
import triton
import triton.language as tl


@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr, INT64: tl.constexpr):
    pid = tl.program_id(0)
    if INT64:
        pid = pid.to(tl.int64)  # offsets past 2**31 need 64-bit indexing
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)


def add(x, y):
    out = torch.empty_like(x)
    n = out.numel()
    block = 1024 if n >= 4096 else 128  # a branch of the host code on the size
    add_kernel[(triton.cdiv(n, block),)](x, y, out, n, BLOCK=block, INT64=n >= 2**31)
    return out

Writing user_code.py


## 2. Triton hook

Here we intercept each Triton launch, introspect which guards apply to the symbolic inputs, and map those inputs to CUDA kernel parameters.

In this example we patch Triton's launcher (`JITFunction.run`). I'll work with the Triton team on the best non-intrusive API for this. Two options:

1. A pre-run hook that takes the arguments (and grid), and can modify and return them.
2. A specialization binder that runs in Python instead of C++.

- `intercept_launches` sends each launch to the tracer.
- `launch` makes a launch through Triton's own `run` and returns the compilation it used. Triton checks that each pointer is device memory, which the trace's addresses need not be, so each is passed as an address in a small scratch buffer the tracer makes before capture, at the same alignment.
- `compile_for` runs Triton's dispatch at given values without launching (`run(warmup=True)`), which compiles the kernel if this specialization is new, and loads it before a capture.
- `specialization` reads from a compilation each argument's type and the checks Triton's binder made on its value to pick the compilation.

In [2]:
%%writefile triton_hook.py
import contextlib
from dataclasses import dataclass

import torch
from triton.runtime.jit import JITFunction


original_run = JITFunction.run  # what kernel[grid](*args) calls


@contextlib.contextmanager
def patch_run(run):
    JITFunction.run = run
    try:
        yield
    finally:
        JITFunction.run = original_run


@dataclass
class Pointer:
    """A tensor argument as Triton reads it."""

    dtype: torch.dtype
    ptr: int

    def data_ptr(self):
        return self.ptr


def launch(jit, args, grid, scratch):
    """Launch with these argument values; returns the compilation Triton used. Triton's launcher
    checks that each pointer is device memory, which a trace's addresses need not be, so each
    Pointer is passed as an address in scratch, a small device buffer, at the same alignment,
    all that Triton specializes on. The capture records that address, and each replay writes over it."""
    args = {k: Pointer(v.dtype, scratch.data_ptr() + v.ptr % 16) if isinstance(v, Pointer) else v for k, v in args.items()}
    return original_run(jit, grid=grid, warmup=False, **args)


def compile_for(jit, args, grid):
    """Triton's dispatch at these argument values, with no launch: the compilation eager would
    pick, compiled if it is new, and loaded."""
    compiled = original_run(jit, grid=grid, warmup=True, **args)
    compiled._init_handles()
    return compiled


# TODO(upstream API): a launch hook that can take the launch over (today: patching JITFunction.run).
def intercept_launches(on_launch):
    """Call on_launch(kernel, args by name, constexpr names, grid) in place of the launch."""

    def run(jit, *args, grid, warmup, **kwargs):
        # grid and warmup are run's own keywords; args and kwargs are the kernel's arguments
        constexprs = [p.name for p in jit.params if p.is_constexpr]
        return on_launch(jit, dict(zip(jit.arg_names, args)) | kwargs, constexprs, grid)

    return patch_run(run)


def specialization(jit, compiled):
    """{name: (type, checks)} per non-constexpr argument: the type Triton compiled it as
    ("constexpr" for an integer equal to 1, compiled in), and the checks on its value
    that picked the compilation, as Triton's binder makes them: "divisible" by 16, and for
    an integer "one" and "i32" (its type is picked by range). Annotations and
    do_not_specialize(_on_alignment) turn checks off."""
    checks = {}
    for p, ty in zip(jit.params, compiled.src.signature.values()):
        if p.is_constexpr:
            continue
        is_int = ty == "constexpr" or (ty[0] in "iu" and ty != "u1")
        specialize = not p.do_not_specialize and p.annotation_type not in ("u1", "fp16", "bf16", "fp32", "fp64")
        checks[p.name] = (ty, [
            *(["divisible"] if specialize and not p.do_not_specialize_on_alignment and (is_int or ty[0] == "*") else []),
            *(["one"] if specialize and is_int else []),
            *(["i32"] if is_int and not p.annotation_type else []),
        ])
    return checks

Writing triton_hook.py


## 3. Tracing

- Each input tensor gets mapped to a fake tensor with symbolic sizes, strides, and a symbolic `data_ptr()`. The sizes have hints; the `data_ptr()` is unbacked, so only its alignment gets guarded.
- A tensor the user code allocates (`aten.empty`, `aten.empty_like`) is caught by a `TorchDispatchMode`, which records its size expressions and gives it a fresh unbacked address symbol (`buf0_ptr`).
- At each launch, we record the grid and arguments as expressions over the input symbols, then launch with the symbolic ones evaluated at this call's inputs, so the capture has a real launch for replay to patch. Each decision Triton makes from those values is recorded as a guard, so no value from this call is used without one.

The graph's guards:
- the user code's decisions on a symbol, such as `block = 1024 if n >= 4096 else 128`, which `ShapeEnv` records with the user's line;
- the launcher's empty-grid check. If it flips, the launch disappears, which changes the graph.

The launch's guards (`dispatch_guards`):
- a constexpr computed from a symbol, since its value is compiled in (`INT64` here);
- Triton's specialization: each check its binder made on an argument (divisible by 16; for an integer, equal to 1 and in i32 range), guarded at its outcome either way, so a replay runs the compilation eager would pick. An annotation that turns a check off (`do_not_specialize`, `do_not_specialize_on_alignment`, a type such as `n: tl.int64`) just leaves out that guard.

In [3]:
%%writefile tracing.py
import os
import weakref
from dataclasses import dataclass

import sympy
import torch
from sympy import And, Eq
from torch._dynamo.source import ConstantSource
from torch._subclasses.fake_tensor import FakeTensor, FakeTensorMode, unset_fake_temporarily
from torch.fx.experimental.symbolic_shapes import DimDynamic, ShapeEnv
from torch.utils._python_dispatch import TorchDispatchMode
from torch.utils._sympy.numbers import int_oo

import triton_hook


@dataclass
class Launch:
    kernel: object  # the JITFunction, whose dispatch a replay can run again
    args: dict  # name -> expression; a tensor's is its address
    dtypes: dict  # name -> dtype, for the tensor arguments
    grid: tuple  # three expressions
    guards: list  # the launch's own: (condition, its value) for what Triton's dispatch decided
    params: list  # (expression, Triton type) per kernel parameter Triton passes from the arguments


@dataclass
class Allocation:
    sizes: list  # expressions
    dtype: torch.dtype
    device: torch.device
    ptr: sympy.Symbol


def symbol(name):
    return sympy.Symbol(name, integer=True, nonnegative=True)


def symbol_values(**tensors):
    """The symbols' values for real tensors."""
    values = {}
    for name, t in tensors.items():
        values.update({symbol(f"{name}_size{d}"): n for d, n in enumerate(t.shape)})
        values[symbol(f"{name}_ptr")] = t.data_ptr()
    return values


def dispatch_guards(kernel, compiled, args, values):
    """What Triton's dispatch decided for a launch, as guards over the argument expressions
    `args`, each with its value at the symbols' `values`, and the kernel parameters of the
    compilation it picked. A constexpr computed from a symbol is compiled in, and so is each
    check Triton's binder made on an argument. Each is guarded at its outcome either way, so
    a replay runs the compilation eager would pick."""
    conditions = []
    for p in kernel.params:
        e = args[p.name]
        if p.is_constexpr and e.free_symbols:  # guard the value, or a bool's condition
            conditions.append(Eq(e, e.subs(values)) if isinstance(e, sympy.Expr) else e)  # a bool's is its condition
    params = []
    for name, (ty, checks) in triton_hook.specialization(kernel, compiled).items():
        e = args[name]
        named = {"divisible": Eq(e % 16, 0), "one": Eq(e, 1), "i32": And(e >= -(2**31), e < 2**31)}
        conditions += [named[check] for check in checks]
        if ty != "constexpr":  # an integer equal to 1 is compiled in: no parameter
            params.append((e, ty))
    return [(c, bool(c.subs(values))) for c in conditions], params


class FakeInput(FakeTensor):
    """A fake input whose data_ptr() is its address symbol (data_ptr is not an ATen op)."""

    ptr: torch.SymInt

    def data_ptr(self):
        return self.ptr


class Allocations(TorchDispatchMode):
    """Make each tensor the eager code allocates a FakeInput with a fresh address symbol.
    This example handles only allocations: any other ATen op raises."""

    def __init__(self, trace):
        super().__init__()
        self.trace = trace

    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        if func.overloadpacket not in (torch.ops.aten.empty, torch.ops.aten.empty_like):
            raise NotImplementedError(f"only allocations are handled, not {func}")
        out = func(*args, **(kwargs or {}))
        trace = self.trace
        name = f"buf{len(trace.allocations)}"
        with unset_fake_temporarily():
            t = FakeInput(trace.mode, torch.empty(out.shape, dtype=out.dtype, device="meta"), out.device)
        # No replay uses the hint (each allocates its own), so it only needs the allocator's alignment.
        t.ptr = trace.address(f"{name}_ptr", 0)
        sizes = [trace.expr(s) for s in out.shape]
        trace.allocations.append(Allocation(sizes, out.dtype, out.device, symbol(f"{name}_ptr")))
        # Weak, as the trace holds every tensor: a strong reference makes the factory
        # (make_variable) return a detached copy, without .ptr.
        trace.allocated.append(weakref.ref(t))
        return t


class Trace:
    def __init__(self, fn, graph, **examples):
        self.hints = {}  # symbol -> its value in this trace's launches: the eager run's, but for allocations
        self.rename = {}  # ShapeEnv symbol -> {name}_size{d} or {name}_ptr, so guards read by name
        self.guards = []  # the graph's: (condition, its value at the hints, where it comes from)
        self.launches = []
        self.allocations = []
        self.allocated = []  # weak references to the FakeInputs the allocations return

        self.shape_env = ShapeEnv()
        self.mode = FakeTensorMode(shape_env=self.shape_env)
        inputs = {}
        for name, t in examples.items():
            sizes = [self.symint(f"{name}_size{d}", n) for d, n in enumerate(t.shape)]
            inputs[name] = FakeInput(self.mode, torch.empty(sizes, dtype=t.dtype, device="meta"), t.device)
            inputs[name].ptr = self.address(f"{name}_ptr", t.data_ptr())
        # For triton_hook.launch. Made before capture, so it is not graph memory.
        self.scratch = torch.empty(16, dtype=torch.uint8, device="cuda")

        with torch.cuda.graph(graph), self.mode, Allocations(self), triton_hook.intercept_launches(self.on_launch):
            result = fn(**inputs)
        # Which allocation fn returns.
        self.output = next(k for k, t in enumerate(self.allocated) if t() is result)
        # The host code's own branches on symbols, which ShapeEnv recorded with the user's frame.
        for g in self.shape_env.guards:
            frame = g.sloc.framework_loc
            self.guards.append((g.expr.xreplace(self.rename), True, f"{os.path.basename(frame.filename)}:{frame.lineno} in {frame.name}"))

    def symint(self, name, hint):
        # >= 0, but not sympy-positive like from_tensor's sizes, so a branch on n == 0 is guarded
        s = self.shape_env.create_unspecified_symint_and_symbol(hint, ConstantSource(name), DimDynamic.DYNAMIC)
        self.shape_env.constrain_symbol_range(s.node.expr, 0, int_oo)
        self.rename[s.node.expr] = symbol(name)
        self.hints[symbol(name)] = hint
        return s

    def address(self, name, value):
        # Unbacked, so host code cannot branch on it. Its only guards are Triton's alignment
        # checks, which on_launch adds.
        s = self.shape_env.create_unbacked_symint()
        self.shape_env.constrain_symbol_range(s.node.expr, 0, int_oo)
        self.rename[s.node.expr] = symbol(name)
        self.hints[symbol(name)] = value
        return s

    def at_hint(self, v):
        if isinstance(v, torch.Tensor):
            return triton_hook.Pointer(v.dtype, self.at_hint(v.data_ptr()))
        if isinstance(v, (torch.SymInt, torch.SymFloat, torch.SymBool)):
            return {torch.SymInt: int, torch.SymFloat: float, torch.SymBool: bool}[type(v)](self.expr(v).subs(self.hints))
        return v

    def expr(self, v):
        if isinstance(v, torch.Tensor):
            v = v.data_ptr()  # Triton passes a tensor as its address
        if isinstance(v, (torch.SymInt, torch.SymFloat, torch.SymBool)):
            return v.node.expr.xreplace(self.rename)
        return sympy.sympify(v)

    def on_launch(self, kernel, args, constexprs, grid):
        # Launch at the hints, as eager did. Triton returns the compilation it used.
        compiled = triton_hook.launch(kernel, {n: self.at_hint(v) for n, v in args.items()}, tuple(map(self.at_hint, grid)), self.scratch)
        exprs = {name: self.expr(v) for name, v in args.items()}
        guards, params = dispatch_guards(kernel, compiled, exprs, self.hints)

        grid = tuple(self.expr(g) for g in (*grid, 1, 1)[:3])  # padded to three dimensions
        # Triton's launcher skips an empty grid. That removes the launch, which changes the
        # graph, so it is a guard of the graph's.
        nonempty = grid[0] * grid[1] * grid[2] > 0
        self.guards.append((nonempty, bool(nonempty.subs(self.hints)), "the launcher's empty-grid check"))
        if self.guards[-1][1]:
            dtypes = {n: v.dtype for n, v in args.items() if isinstance(v, torch.Tensor)}
            self.launches.append(Launch(kernel, exprs, dtypes, grid, guards, params))

Writing tracing.py


## 4. Runtime

The tracer captures the user code into `Replay`'s graph, so the graph holds exactly the launches the trace made, with every kernel parameter set by Triton's own launcher. A call:

1. allocates what the user code allocates, at the sizes evaluated on the new inputs;
2. checks the graph's guards; if one fails, the replay misses;
3. picks the launch's first entry whose guards hold. If none holds, it runs the launch's dispatch again at the call's values (`Replay.redispatch`): Triton picks or compiles the kernel eager would run, and its launch is captured alone, which gives a new entry;
4. writes the entry's function, its parameters that have expressions, and the grid into the graph's kernel node, and replays.

Each parameter is written in the encoding of the type Triton compiled it as (an address, an integer, a float or a bool). Only the symbols are guarded: the inputs are assumed contiguous, with the examples' dtype, rank and device.

`DynamicCudaGraph` tries its replays in order and returns the first hit's output. If none hits, it traces again at the call's inputs (the warm-up gives the call's result). It records why each trace after the first happened (`retrace_causes`): the guard that failed, and where it came from.

Our implementation does the same per op: a failing op guard runs that op's dispatch again and adds its launches as an entry selected by the op's guards. Where the dispatch gives different outputs, allocations or a different number of launches, or takes an argument it can't evaluate at the call (in our current snapshot, a symbolic bool or float), it refuses, and the call is traced again.

In [4]:
%%writefile replay.py
import ctypes
import struct

import sympy
import torch
from cuda.bindings import driver
from torch.cuda._utils import _check_cuda_bindings

import triton_hook
from tracing import dispatch_guards, Trace, symbol_values


def to_bytes(value, ty, size):
    """value as Triton passes a parameter of type ty ("*fp32", "i32", "u64", "fp32", "u1", ...)."""
    if ty.startswith("fp"):
        return struct.pack({2: "<e", 4: "<f", 8: "<d"}[size], float(value))
    return int(bool(value) if ty == "u1" else value).to_bytes(size, "little", signed=ty[0] == "i")


class Entry:
    """One compilation of the launch: the guards under which Triton's dispatch picks it, and
    the kernel node's function and parameters for it, read from a capture of the launch made
    by Triton's own launcher (which also sets the parameters it appends after the arguments)."""

    def __init__(self, graph, guards, params):
        self.graph, self.guards, self.params = graph, guards, params  # the graph is kept for its node
        (self.node,), _ = _check_cuda_bindings(driver.cuGraphGetNodes(graph.raw_cuda_graph(), 1))
        p = _check_cuda_bindings(driver.cuGraphKernelNodeGetParams(self.node))
        # Each parameter's size. cuFuncGetParamInfo returns (error, offset, size)
        # and errors past the last parameter.
        sizes = []
        while (info := driver.cuFuncGetParamInfo(p.func, len(sizes)))[0] == driver.CUresult.CUDA_SUCCESS:
            sizes.append(info[2])
        # Writable copies of the captured parameter values, which each call
        # overwrites, passed as kernelParams: an array of one pointer per parameter.
        captured = (ctypes.c_void_p * len(sizes)).from_address(int(p.kernelParams))
        self.buffers = [ctypes.create_string_buffer(ctypes.string_at(ptr, size), size) for ptr, size in zip(captured, sizes)]
        self.pointers = (ctypes.c_void_p * len(sizes))(*map(ctypes.addressof, self.buffers))
        p.kernelParams = ctypes.addressof(self.pointers)
        self.node_params = p

    def holds(self, values):
        return next(((c, v) for c, v in self.guards if bool(c.subs(values)) != v), None)


class Replay:
    def __init__(self, fn, **tensors):
        self.graph = torch.cuda.CUDAGraph(keep_graph=True)  # keep the graph to read its node
        trace = Trace(fn, self.graph, **tensors)
        self.graph.instantiate()
        self.guards = trace.guards
        self.allocations, self.output = trace.allocations, trace.output
        (self.launch,) = trace.launches  # this example records one launch
        self.entries = [Entry(self.graph, self.launch.guards, self.launch.params)]
        self.node = self.entries[0].node  # the graph's one node is the launch
        self.scratch = trace.scratch

    def redispatch(self, values):
        """Run the launch's dispatch again, at the call's values: Triton picks (or compiles) the
        kernel eager would run. Its launch is captured alone, which gives a new entry. No user
        code runs, and nothing else is traced."""
        def at(e):  # an expression's value at the call (a sympy Symbol is also a Boolean: test Expr first)
            if isinstance(e, sympy.Expr):
                return int(e.subs(values)) if e.is_integer else float(e.subs(values))
            return bool(e.subs(values))

        launch = self.launch
        args = {}
        for p in launch.kernel.params:
            e = launch.args[p.name]
            # a tensor argument is an address symbol, passed as a pointer of the traced dtype
            args[p.name] = triton_hook.Pointer(launch.dtypes[p.name], at(e)) if p.name in launch.dtypes else at(e)
        grid = tuple(at(g) for g in launch.grid)
        triton_hook.compile_for(launch.kernel, args, grid)  # outside the capture, which can't load a module
        graph = torch.cuda.CUDAGraph(keep_graph=True)
        with torch.cuda.graph(graph):
            compiled = triton_hook.launch(launch.kernel, args, grid, self.scratch)
        guards, params = dispatch_guards(launch.kernel, compiled, launch.args, values)
        self.entries.append(Entry(graph, guards, params))
        return self.entries[-1]

    def __call__(self, **tensors):
        values = symbol_values(**tensors)
        # Allocate as eager would, before the guards, which can involve the addresses.
        allocated = []
        for a in self.allocations:
            allocated.append(torch.empty([int(s.subs(values)) for s in a.sizes], dtype=a.dtype, device=a.device))
            values[a.ptr] = allocated[-1].data_ptr()
        for cond, expected, where in self.guards:
            if bool(cond.subs(values)) != expected:
                return f"miss: ({cond}) is {not expected}, from {where}", None
        # The graph's guards hold. The first entry whose guards hold is the compilation eager
        # would pick; if none does, the launch's dispatch runs again.
        how = "hit"
        k = next((k for k, e in enumerate(self.entries) if e.holds(values) is None), None)
        if k is None:
            failed = self.entries[0].holds(values)
            entry = self.redispatch(values)
            k = len(self.entries) - 1
            how = f"{self.launch.kernel.__name__} dispatched again ({failed[0]} is {not failed[1]})"
        entry = self.entries[k]
        # Parameters Triton appends after the arguments have no expression (zip
        # stops there), so they keep their captured bytes.
        for buf, (e, ty) in zip(entry.buffers, entry.params):
            buf.raw = to_bytes(e.subs(values), ty, len(buf))
        p = entry.node_params
        p.gridDimX, p.gridDimY, p.gridDimZ = (int(g.subs(values)) for g in self.launch.grid)
        # The node takes the entry's function and parameters.
        _check_cuda_bindings(driver.cuGraphExecKernelNodeSetParams(self.graph.raw_cuda_graph_exec(), self.node, p))
        self.graph.replay()
        return f"{how}, entry {k}", allocated[self.output]


class DynamicCudaGraph:
    """Call fn through CUDA graphs. A call replays the first graph whose guards hold. Otherwise it
    traces fn at its inputs, as another replay: the trace first runs fn once as a warm-up, which
    compiles what these inputs need and gives the call's result, then runs fn again on symbolic
    inputs under capture."""

    def __init__(self, fn):
        self.fn = fn
        self.replays = []
        self.retrace_causes = []  # per trace after the first: the guard that failed, and where it came from

    def __call__(self, **tensors):
        """Returns fn's result, and how the call ran."""
        misses = []
        for i, replay in enumerate(self.replays):
            how, result = replay(**tensors)
            if result is not None:
                return result, f"replay {i} {how}"
            misses.append(f"replay {i} {how}")
        result = self.fn(**tensors)  # the trace's warm-up
        if self.replays:
            self.retrace_causes.append(misses[0].split(": ", 1)[1])
        self.replays.append(Replay(self.fn, **tensors))
        return result, "; ".join(misses + [f"traced as replay {len(self.replays) - 1}"])

Writing replay.py


## 5. Calls

The first call traces as replay 0. No guard pins `n`.

- The graph's guards are the user code's branch, `x_size0 >= 4096`, with its line, and the launcher's empty-grid check.
- The launch's guards are the `INT64` constexpr's condition, then Triton's specialization checks, each at its outcome: each address divisible by 16, and `n` divisible by 16, not equal to 1, and in i32 range (so it is passed as an i32). The range guards print without their lower bound because the symbols are nonnegative.
- The parameters are the three addresses, `out`'s being the allocation's symbol `buf0_ptr`, and `n`, which is `x_size0` since `out` has `x`'s size; the constexprs are compiled in.

In [5]:
import importlib, torch
import user_code, triton_hook, tracing, replay
for m in (user_code, triton_hook, tracing, replay):
    importlib.reload(m)

torch.manual_seed(0)
x, y = torch.randn(2, 8192, device="cuda")
f = replay.DynamicCudaGraph(user_code.add)

def call(name, xi, yi):
    out, how = f(x=xi, y=yi)
    torch.cuda.synchronize()
    print(f"{name}: {how}; out == x + y: {torch.equal(out, xi + yi)}")

call("n=4096", x[:4096], y[:4096])
r0 = f.replays[0]
print("the graph's guards:")
for cond, value, where in r0.guards:
    print(f"  ({cond}) is {value}, from {where}")
print("the launch's guards (entry 0):")
for cond, value in r0.launch.guards:
    print(f"  ({cond}) is {value}")
print(f"params: {[e for e, _ in r0.launch.params]}")
print(f"grid: {r0.launch.grid}")

n=4096: traced as replay 0; out == x + y: True
the graph's guards:
  ((((x_size0 + 1023)//1024)) > 0) is True, from the launcher's empty-grid check
  (x_size0 >= 4096) is True, from user_code.py:21 in add
the launch's guards (entry 0):
  (x_size0 >= 2147483648) is False
  (Eq(Mod(x_ptr, 16), 0)) is True
  (Eq(Mod(y_ptr, 16), 0)) is True
  (Eq(Mod(buf0_ptr, 16), 0)) is True
  (Eq(Mod(x_size0, 16), 0)) is True
  (Eq(x_size0, 1)) is False
  (x_size0 < 2147483648) is True
params: [x_ptr, y_ptr, buf0_ptr, x_size0]
grid: (((x_size0 + 1023)//1024), 1, 1)


### A specialization flip: the launch is dispatched again

n=8192 and n=4112 hit entry 0. n=4097 is not divisible by 16, which is the launch's own guard, so only the launch is dispatched again: Triton compiles the kernel for an `n` not divisible by 16, as eager would, and it becomes entry 1 of replay 0. No user code runs and nothing is traced again. n=4100 then hits entry 1, and n=4096 entry 0.

In [6]:
call("n=8192", x[:8192], y[:8192])
call("n=4112", x[:4112], y[:4112])
call("n=4097", x[:4097], y[:4097])
call("n=4100", x[:4100], y[:4100])
call("n=4096", x[:4096], y[:4096])

n=8192: replay 0 hit, entry 0; out == x + y: True
n=4112: replay 0 hit, entry 0; out == x + y: True
n=4097: replay 0 add_kernel dispatched again (Eq(Mod(x_size0, 16), 0) is False), entry 1; out == x + y: True
n=4100: replay 0 hit, entry 1; out == x + y: True
n=4096: replay 0 hit, entry 0; out == x + y: True


### Alignment flips

A misaligned `x` fails entry 0's and entry 1's alignment guard on `x`, so the launch is dispatched again, as entry 2, compiled for a misaligned `x`; other misaligned `x` then hit it. A misaligned `y` fails entry 2 too, which requires `x` misaligned and `y` aligned, and becomes entry 3. Since every check is guarded both ways, a call only runs the compilation eager would pick: aligned inputs run entry 0, never entry 2, whatever the order.

In [7]:
call("x[1:4097], n=4096", x[1:4097], y[:4096])
call("x[3:4099], n=4096", x[3:4099], y[:4096])
call("y[1:4097], n=4096", x[:4096], y[1:4097])
call("n=4096", x[:4096], y[:4096])
print()
print("entry 0's and entry 2's guards differ on:")
e0, e2 = r0.entries[0].guards, dict(r0.entries[2].guards)
for cond, value in e0:
    if e2.get(cond) != value:
        print(f"  ({cond}): entry 0 {value}, entry 2 {e2.get(cond)}")

x[1:4097], n=4096: replay 0 add_kernel dispatched again (Eq(Mod(x_ptr, 16), 0) is False), entry 2; out == x + y: True
x[3:4099], n=4096: replay 0 hit, entry 2; out == x + y: True
y[1:4097], n=4096: replay 0 add_kernel dispatched again (Eq(Mod(y_ptr, 16), 0) is False), entry 3; out == x + y: True
n=4096: replay 0 hit, entry 0; out == x + y: True

entry 0's and entry 2's guards differ on:
  (Eq(Mod(x_ptr, 16), 0)): entry 0 True, entry 2 False


### A branch in the user code: traced again

n=1000 fails the graph's guard `x_size0 >= 4096`, which the user code's Python recorded: with n=1000 the user code takes the other branch, so the call is traced again as replay 1, and the cause names the guard and the user's line. n=500 hits replay 1. n=1 is equal to 1, a check of Triton's binder, so replay 1's launch is dispatched again: Triton compiles `n` in as a constant, and the new entry has one parameter fewer. n=1008 is divisible by 16, another flip of the launch's own guards.

In [8]:
call("n=1000", x[:1000], y[:1000])
call("n=500", x[:500], y[:500])
call("n=1", x[:1], y[:1])
call("n=1008", x[:1008], y[:1008])
print()
print(f"traces: {len(f.replays)}; retrace causes: {f.retrace_causes}")
for i, r in enumerate(f.replays):
    print(f"replay {i}: {len(r.entries)} entries, with {[len(e.params) for e in r.entries]} parameters")

n=1000: replay 0 miss: (x_size0 >= 4096) is False, from user_code.py:21 in add; traced as replay 1; out == x + y: True
n=500: replay 1 hit, entry 0; out == x + y: True
n=1: replay 1 add_kernel dispatched again (Eq(x_size0, 1) is True), entry 1; out == x + y: True
n=1008: replay 1 add_kernel dispatched again (Eq(Mod(x_size0, 16), 0) is True), entry 2; out == x + y: True

traces: 2; retrace causes: ['(x_size0 >= 4096) is False, from user_code.py:21 in add']
replay 0: 4 entries, with [4, 4, 4, 4] parameters
replay 1: 3 entries, with [4, 3, 4] parameters


## 6. Our implementation on the same calls

The same calls through our implementation's entry point, `HostTraceReplay` (a private API in our build). It differs from the example above in a few ways:

- its first call runs eagerly and its second call traces (not the first);
- it handles allocations, views and ATen ops, and many launches per graph;
- `redispatches` counts the launches dispatched again, `traces` the traces;
- `retrace_causes` maps each reason for a trace after the first, as (class, op, guard, the user's line, a redispatch's refusal), to its count. Class `graph` is a guard no op recorded, such as the branch in the user code; class `dispatch` is a launch's own guard whose redispatch was refused.

We run it twice: on `user_code.add`, and on a copy with `INT64=False` instead of `INT64=n >= 2**31`. In our current snapshot a redispatch refuses a launch that takes a symbolic bool, which `n >= 2**31` is, so for `user_code.add` each specialization flip is traced again, with that refusal as its cause. Without it, the flips are dispatched again, as in the example above. Each output is compared with eager's (`x + y`) with `torch.equal`, which is a bitwise comparison here.

In [9]:
import triton
from torch.cuda._host_trace_replay import HostTraceReplay

def add_i32(x, y):  # user_code.add with INT64=False: no symbolic bool argument
    out = torch.empty_like(x)
    n = out.numel()
    block = 1024 if n >= 4096 else 128
    user_code.add_kernel[(triton.cdiv(n, block),)](x, y, out, n, BLOCK=block, INT64=False)
    return out

calls = [(f"n={n}", x[:n], y[:n]) for n in (4096, 4096, 8192, 4112, 4097, 4100)]
calls += [("x[1:4097]", x[1:4097], y[:4096]), ("x[3:4099]", x[3:4099], y[:4096]), ("y[1:4097]", x[:4096], y[1:4097]), ("n=4096", x[:4096], y[:4096])]
calls += [(f"n={n}", x[:n], y[:n]) for n in (1000, 500, 1, 1008)]
for fn in (user_code.add, add_i32):
    h = HostTraceReplay(fn)
    bitwise = True
    for name, xi, yi in calls:
        out = h(xi, yi)
        torch.cuda.synchronize()
        bitwise &= torch.equal(out, xi + yi)
    print(f"{fn.__name__}, {len(calls)} calls: traces {h.traces}, replays {h.replays}, redispatches {h.redispatches}, eager {h.eager}; every output equal to eager's: {bitwise}")
    for (cls, op, guard, line, refusal), count in h.retrace_causes.items():
        print(f"  {count} retrace: class {cls}, guard {guard}, at {line}, refusal {refusal}")

add, 14 calls: traces 7, replays 6, redispatches 0, eager 1; every output equal to eager's: True
  1 retrace: class dispatch, guard ((s61 % 16) == 0), at backends/compiler.py:84, refusal JITFunction(user_code:add_kernel) takes a symbolic SymBool
  1 retrace: class dispatch, guard (((4*s88 + s8) % 16) == 0), at backends/compiler.py:90, refusal JITFunction(user_code:add_kernel) takes a symbolic SymBool
  1 retrace: class dispatch, guard (((4*s34 + s38) % 16) == 0), at backends/compiler.py:90, refusal JITFunction(user_code:add_kernel) takes a symbolic SymBool
  1 retrace: class graph, guard ((-1*s61 + 4096) <= 0), at run_20261008/user_code.py:21, refusal None
  1 retrace: class dispatch, guard ((s61 + -1) != 0), at /<string>:4, refusal JITFunction(user_code:add_kernel) takes a symbolic SymBool
  1 retrace: class dispatch, guard ((s61 % 16) != 0), at backends/compiler.py:84, refusal JITFunction(user_code:add_kernel) takes a symbolic SymBool
add_i32, 14 calls: traces 2, replays 11, redispat

## 7. How this relates to the rest of our implementation

- **Who owns a guard.** A guard an op's dispatch records (here Triton's specialization of a launch) belongs to that op: a flip dispatches the op again, as above, unless the redispatch refuses (section 6). A guard the Python around the ops records belongs to the graph: a flip traces again, and `retrace_causes` says which guard, which op recorded it, and which user line. An earlier mechanism that re-derived downstream metadata after such a change has been removed.
- **What is traced and what is measured.** Triton, CuTe DSL and ATen launches are traced symbolically, guarded, or declined (run eagerly, with the reason). Only closed-source kernels, whose host code we can't trace (cuBLAS/cuBLASLt, cuDNN, closed attention kernels), have their launches learned from captures at each new key, which is checked empirically rather than sound by construction. There is no budget on how many keys are learned, and a cuBLAS call always gets its workspace, as in eager.
- **Hint reads.** The goal is that no trace makes a decision from an example value without recording a guard for it. This example meets it for its one launch: every check Triton made at the example's values is a guard.